# Kikuchi Guiding State Tutorial

Complete native counterpart of the frozen Qualtran v0.7.0 tutorial at
`bcb17bb25360ed68ce3bc084645db3e929799389`, blob
`34e5a813e1b5eaa1f5a9f5774f145be5833c789d`. Copyright 2024 Google LLC,
adapted under Apache-2.0. Diagrams retain logical calls; QEC estimates are
separate from emitted primitives. Scope preparation and sorting have no
original physical bodies, so these diagrams do not certify full preparation.
Concrete duplicate detection retains the source inverse-alias defect.
See `docs/development/KIKUCHI_GUIDING_STATE_FILE_AUDIT.md`.


In [ ]:
from collections import Counter
from collections.abc import Callable
from html import escape

import numpy as np
import sympy
from IPython.display import SVG, display

from flagquantum.algorithms.kikuchi_guiding_resource_graph import (
    guiding_call_title,
    source_guiding_call_graph,
)
from flagquantum.algorithms.kikuchi_guiding_state import (
    GuidingState,
    ProbabilisticUncompute,
)
from flagquantum.algorithms.kxor_instance import KXorInstance
from flagquantum.compiler._component_trace.gate_budget import t_and_ccz_budget
from flagquantum.compiler._component_trace.symbolic_math import asymptotic_order
from flagquantum.compiler.registers.component_expansion import (
    expand_component_calls_once,
)
from flagquantum.core.ir.program import FunctionRecord, ProgramRecord
from flagquantum.core.ir.program.registers import RegisterSide
from flagquantum.core.value_math import ceil, floor, log2
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.component_graph import export_component_graph_dot
from flagquantum.drawer.signature_graph import export_signature_graph_dot
from flagquantum.dynamic import DynamicCircuit

make_svg: Callable[[bytes], SVG] = SVG
show_image: Callable[[object], None] = display


def component_title(function: FunctionRecord) -> str:
    registers = function.registers or ()
    ports = tuple(register.name for register in registers)
    if "x" in ports and len(ports) > 1:
        return "Partition"
    titles = {
        ("S",): "Scope guiding state",
        ("T", "ancilla"): "Packed sort",
        ("T", "flag"): "Packed duplicate check",
        ("xs", "pi"): "Sort in place",
        ("xs", "flag"): "Duplicate check",
        ("q", "flag"): "Probabilistic uncompute",
        ("controls", "target"): "Multi-control X",
        ("ctrl", "q"): "Controlled X",
    }
    if ports == ("q",):
        return "X" if registers[0].bitsize == 1 else "H on word"
    return titles.get(ports, "Component")


def show_program(program: ProgramRecord) -> None:
    # Rename display labels only; stored calls and formal ports remain intact.
    dot = export_component_graph_dot(program, pretty=False)
    for function in sorted(program.functions, key=lambda f: len(f.name), reverse=True):
        dot = dot.replace(escape(function.name), escape(component_title(function)))
    show_image(make_svg(render_call_graph_svg(dot)))


def show_call_graph(guide: GuidingState, *, max_depth: int) -> None:
    graph = source_guiding_call_graph(guide, max_depth=max_depth)
    dot = export_call_graph_dot(
        graph.root, graph.direct_calls, title=guiding_call_title
    )
    show_image(make_svg(render_call_graph_svg(dot)))


Let us start with a kXOR instance with $n$ variables and $m$ constraints.

In [ ]:
n, m, k = sympy.symbols("n m k", positive=True, integer=True)
inst = KXorInstance.symbolic(n, m, k)
inst


We first prepare the guiding state to use in the guided sparse hamiltonian algorithm.
The guiding state is defined by the instance, and a parameter $\ell$ (a multiple of $k$)

From Theorem 4.15 of the paper, this should be a circuit of $O(\ell m \log n)$ gates,
and prepare the state $\beta |\Psi\rangle|0^{\ell \log \ell}\rangle + |\perp\rangle|1\rangle$,
where $\beta \ge 0.99 / \ell^{\ell/2}$.

In [ ]:
c = sympy.symbols("c", positive=True, integer=True)
ell = c * k
guiding_state = GuidingState(inst, ell)
show_call_graph(guiding_state, max_depth=1)


In [ ]:
guiding_state_3 = GuidingState(inst, 3 * k)
symbolic_root = guiding_state_3.program()
show_program(symbolic_root)


We can also build the guiding state for a concrete (non symbolic) instance:

In [ ]:
inst = KXorInstance.random_instance(
    n=20, m=100, k=4, planted_advantage=0.8, rng=np.random.default_rng(100)
)
guiding_state_concrete = GuidingState(inst, ell=12)
show_image(
    make_svg(
        render_call_graph_svg(
            export_signature_graph_dot(
                guiding_state_concrete.signature, "Guiding state", pretty=True
            )
        )
    )
)


In [ ]:
concrete_root = guiding_state_concrete.program()
show_program(concrete_root)


In [ ]:
functions = {function.name: function for function in concrete_root.functions}
# Upstream Partition is atomic for flatten_once, despite native zero-gate lowering.
flattened_root, instance_labels = expand_component_calls_once(
    concrete_root,
    predicate=lambda _, operation: all(
        register.side == RegisterSide.THRU
        for register in functions[operation.callee or ""].registers or ()
    ),
)
show_program(flattened_root)


Let us evaluate the gate cost for the above bloqs.

In [ ]:
concrete_cost = guiding_state_concrete.resource_gates()
concrete_cost


In [ ]:
gc = guiding_state.resource_gates()
budget = t_and_ccz_budget(gc, ts_per_rotation=11)
t_cost = sympy.sympify(budget["n_t"] + 4 * budget["n_ccz"])
raw_t_cost = t_cost
t_cost


In [ ]:
klogn = k * ceil(log2(n))
klogn_long = ceil(log2(floor(2**klogn)))
t_cost = t_cost.subs(klogn_long, klogn).simplify()
l_symb = sympy.symbols(r"\ell", positive=True, integer=True)
t_cost = t_cost.subs(c * k, l_symb)
declared_order = asymptotic_order(t_cost)
declared_order


In [ ]:
show_call_graph(guiding_state_concrete, max_depth=3)


As we know that $c = \ell/k \le \ell$ and $\ell \le m$, the above expression matches the paper result of $O(\ell m \log_2(n))$ 1/2-qubit gates.
<!-- Here, we synthesize each 1-qubit rotation to precision $2^{-b_\text{grad}}$, with $4 (b_\text{grad}-2)$ T gates using phase gradients. -->

In [ ]:
# Separate actual emission from the symbolic/full-root model above.

op = ProbabilisticUncompute(2).bind((0, 1), 2, (3,), (0,))
circuit = DynamicCircuit(4)
circuit.append(op)
measured_primitives = Counter(gate.name for gate in circuit.to_ir().instructions)
uncompute_model = ProbabilisticUncompute(2).resource_gates()
print("Actual uncompute primitive counts:", dict(measured_primitives))
print("Separate uncompute QEC model:", uncompute_model)
